In [ ]:
# Databricks notebook source
import sys
sys.path.append("../data_quality")
from dq_utils import ensure_log_table_exists, check_not_null, check_unique, check_no_orphans

spark.sql("USE CATALOG brazilian_ecommerce")
ensure_log_table_exists(spark)

staging_customers = spark.table("prod.staging_customers")
staging_sellers = spark.table("prod.staging_sellers")
staging_orders = spark.table("prod.staging_orders")
staging_order_items = spark.table("prod.staging_order_items")
staging_order_payments = spark.table("prod.staging_order_payments")
staging_order_reviews = spark.table("prod.staging_order_reviews")
staging_products = spark.table("prod.staging_products")
staging_category_translation = spark.table("prod.staging_category_translation")
staging_geolocation = spark.table("prod.staging_geolocation")

# --- customers ---
check_not_null(spark, "staging", "staging_customers", staging_customers, "customer_id", raise_on_failure=True)
check_unique(spark, "staging", "staging_customers", staging_customers, ["customer_id"], raise_on_failure=True)

# --- sellers ---
check_not_null(spark, "staging", "staging_sellers", staging_sellers, "seller_id", raise_on_failure=True)
check_unique(spark, "staging", "staging_sellers", staging_sellers, ["seller_id"], raise_on_failure=True)

# --- orders ---
check_not_null(spark, "staging", "staging_orders", staging_orders, "order_id", raise_on_failure=True)
check_unique(spark, "staging", "staging_orders", staging_orders, ["order_id"], raise_on_failure=True)
check_no_orphans(spark, "staging", "staging_orders", staging_orders, "customer_id", staging_customers, "customer_id", raise_on_failure=True)

# --- order items ---
check_unique(spark, "staging", "staging_order_items", staging_order_items, ["order_id", "order_item_id"], raise_on_failure=True)
check_no_orphans(spark, "staging", "staging_order_items", staging_order_items, "order_id", staging_orders, "order_id", raise_on_failure=True)
check_no_orphans(spark, "staging", "staging_order_items", staging_order_items, "product_id", staging_products, "product_id", raise_on_failure=True)
check_no_orphans(spark, "staging", "staging_order_items", staging_order_items, "seller_id", staging_sellers, "seller_id", raise_on_failure=True)

# --- order payments ---
check_unique(spark, "staging", "staging_order_payments", staging_order_payments, ["order_id", "payment_sequential"], raise_on_failure=True)
check_no_orphans(spark, "staging", "staging_order_payments", staging_order_payments, "order_id", staging_orders, "order_id", raise_on_failure=True)

# --- order reviews ---
check_not_null(spark, "staging", "staging_order_reviews", staging_order_reviews, "review_id", raise_on_failure=True)
check_unique(spark, "staging", "staging_order_reviews", staging_order_reviews, ["review_id"], raise_on_failure=True)
check_no_orphans(spark, "staging", "staging_order_reviews", staging_order_reviews, "order_id", staging_orders, "order_id", raise_on_failure=True)

# --- products ---
check_not_null(spark, "staging", "staging_products", staging_products, "product_id", raise_on_failure=True)
check_unique(spark, "staging", "staging_products", staging_products, ["product_id"], raise_on_failure=True)

# --- category translation ---
check_not_null(spark, "staging", "staging_category_translation", staging_category_translation, "product_category_name", raise_on_failure=True)
check_unique(spark, "staging", "staging_category_translation", staging_category_translation, ["product_category_name"], raise_on_failure=True)

# --- geolocation (no PK — zip prefix legitimately repeats, see staging build notes) ---
check_not_null(spark, "staging", "staging_geolocation", staging_geolocation, "geolocation_zip_code_prefix")